In [ ]:
import pandas as pd
import numpy as np

import wittgenstein as lw

print("Libraries imported successfully.")



FILE_NAME = "Zoo.csv"

df = pd.read_csv(FILE_NAME)

print("\n================ DATASET ================")

print(df.head())

print("\nShape:", df.shape)

print("\nColumns:")
print(list(df.columns))



TARGET = "type"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("\nTarget column:", TARGET)

print("\nTarget values:")
print(y.unique())

print("\nClass distribution:")
print(y.value_counts())



print("\n\n============================================================")
print("                 RIPPER CLASSIFICATION")
print("============================================================")

classes = df[TARGET].unique()

ripper_models = {}

for cls in classes:

    print("\n------------------------------------------------------------")
    print("RIPPER RULES FOR:", cls)
    print("------------------------------------------------------------")

    ripper = lw.RIPPER(
        random_state=42,
        verbosity=0
    )


    ripper.fit(
        df,
        class_feat=TARGET,
        pos_class=cls
    )

    ripper_models[cls] = ripper

    print(ripper.out_model())



print("\n\n============================================================")
print("                    CN2 CLASSIFICATION")
print("============================================================")

import Orange



cn2_df = df.copy()

for column in cn2_df.columns:
    cn2_df[column] = cn2_df[column].astype(str)



feature_names = [
    column
    for column in cn2_df.columns
    if column != TARGET
]

orange_features = []

for column in feature_names:

    values = sorted(
        cn2_df[column].unique().tolist()
    )

    variable = Orange.data.DiscreteVariable(
        column,
        values=values
    )

    orange_features.append(variable)



target_values = sorted(
    cn2_df[TARGET].unique().tolist()
)

orange_target = Orange.data.DiscreteVariable(
    TARGET,
    values=target_values
)



domain = Orange.data.Domain(
    orange_features,
    orange_target
)



rows = []

for _, row in cn2_df.iterrows():

    values = []

    for column in feature_names:
        values.append(
            str(row[column])
        )

    values.append(
        str(row[TARGET])
    )

    rows.append(values)



data = Orange.data.Table.from_list(
    domain,
    rows
)


print("\nOrange dataset created.")

print("Number of instances:", len(data))

print("Number of features:", len(feature_names))



cn2_learner = Orange.classification.rules.CN2Learner()

cn2_model = cn2_learner(data)



print("\n\n================ CN2 RULES ================\n")

for i, rule in enumerate(
    cn2_model.rule_list,
    start=1
):

    print(
        f"Rule {i}: {rule}"
    )



print("\n\n============================================================")
print("                     FOIL ALGORITHM")
print("============================================================")



def entropy(positive, negative):

    total = positive + negative

    if total == 0:
        return 0

    p_positive = positive / total
    p_negative = negative / total

    result = 0

    if p_positive > 0:
        result -= (
            p_positive *
            np.log2(p_positive)
        )

    if p_negative > 0:
        result -= (
            p_negative *
            np.log2(p_negative)
        )

    return result



def information_gain(
    old_positive,
    old_negative,
    new_positive,
    new_negative
):

    old_total = (
        old_positive +
        old_negative
    )

    new_total = (
        new_positive +
        new_negative
    )

    if old_total == 0 or new_total == 0:
        return 0

    old_entropy = entropy(
        old_positive,
        old_negative
    )

    new_entropy = entropy(
        new_positive,
        new_negative
    )

    return old_entropy - new_entropy



def generate_literals(
    data,
    target,
    current_rule
):

    literals = []

    for column in data.columns:

        if column == target:
            continue

    
        if any(
            literal[0] == column
            for literal in current_rule
        ):
            continue

        for value in data[column].unique():

            literals.append(
                (column, value)
            )

    return literals



def apply_rule(
    data,
    rule
):

    result = data.copy()

    for column, value in rule:

        result = result[
            result[column] == value
        ]

    return result



def foil(
    data,
    target,
    positive_class
):

    rules = []


    positive_examples = data[
        data[target] == positive_class
    ].copy()


    negative_examples = data[
        data[target] != positive_class
    ].copy()






    while len(positive_examples) > 0:

        rule = []

        current_data = data.copy()


    
    
    

        while True:

            positives = current_data[
                current_data[target] == positive_class
            ]

            negatives = current_data[
                current_data[target] != positive_class
            ]


        
            if len(negatives) == 0:
                break


        
            if len(positives) == 0:
                break


            candidates = generate_literals(
                current_data,
                target,
                rule
            )


            if len(candidates) == 0:
                break


            best_literal = None
            best_gain = -float("inf")


        
        
        

            old_positive = len(positives)
            old_negative = len(negatives)


            for literal in candidates:

                column, value = literal

                filtered = current_data[
                    current_data[column] == value
                ]


                new_positive = len(
                    filtered[
                        filtered[target] ==
                        positive_class
                    ]
                )


                new_negative = len(
                    filtered[
                        filtered[target] !=
                        positive_class
                    ]
                )


                if new_positive == 0:
                    continue


                gain = information_gain(
                    old_positive,
                    old_negative,
                    new_positive,
                    new_negative
                )


                if gain > best_gain:

                    best_gain = gain
                    best_literal = literal


        
            if best_literal is None:
                break


        
            rule.append(best_literal)


        
            column, value = best_literal

            current_data = current_data[
                current_data[column] == value
            ]


    
    
    

        if len(rule) == 0:
            break


        rules.append(rule)


    
    
    

        covered = apply_rule(
            positive_examples,
            rule
        )


        if len(covered) == 0:
            break


        positive_examples = positive_examples.drop(
            covered.index
        )


    return rules



print("\n\n================ FOIL RULES ================\n")


for cls in classes:

    print("\n------------------------------------------------------------")
    print("FOIL RULES FOR:", cls)
    print("------------------------------------------------------------")


    rules = foil(
        df,
        TARGET,
        cls
    )


    if len(rules) == 0:

        print("No rules generated.")

        continue


    for i, rule in enumerate(
        rules,
        start=1
    ):

        conditions = []

        for column, value in rule:

            conditions.append(
                f"{column} = {value}"
            )


        condition_string = " AND ".join(
            conditions
        )


        print(
            f"Rule {i}: IF "
            f"{condition_string} "
            f"THEN {TARGET} = {cls}"
        )



print("\n\n============================================================")
print("                       SUMMARY")
print("============================================================")

print("""
Dataset:
UCI Zoo Dataset

Number of instances:
101

Number of features:
16

Number of classes:
7

Algorithms implemented:
1. RIPPER
2. CN2
3. FOIL

Experiment completed successfully.
""")

Libraries imported successfully.

================ DATASET ================
    hair  feathers   eggs   milk  airborne  aquatic  predator  toothed  \
0   True     False  False   True     False    False      True     True   
1   True     False  False   True     False    False     False     True   
2  False     False   True  False     False     True      True     True   
3   True     False  False   True     False    False      True     True   
4   True     False  False   True     False    False      True     True   

   backbone  breathes  venomous   fins  legs   tail  domestic  catsize    type  
0      True      True     False  False     4  False     False     True  mammal  
1      True      True     False  False     4   True     False     True  mammal  
2      True     False     False   True     0   True     False    False    fish  
3      True      True     False  False     4  False     False     True  mammal  
4      True      True     False  False     4   True     False     True  ma